# Synova AI Fine-Tuning Pipeline
## Fine-tune Llama 3.1 8B on Synova-specific dataset

## Setup - Install Required Libraries

In [ ]:
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

packages = ["torch", "transformers", "peft", "datasets", "bitsandbytes", "trl", "accelerate", "wandb"]

for package in packages:
    try:
        __import__(package.replace("-", "_"))
        print(f"{package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        install_package(package)

## Import Libraries

In [ ]:
import torch
import json
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

MODEL_NAME = os.getenv("MODEL_NAME", "meta-llama/Llama-3.2-3B")  # Smaller model for local training
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "./synova-finetuned")
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "2"))  # Smaller batch for memory
GRADIENT_ACCUMULATION_STEPS = int(os.getenv("GRADIENT_ACCUMULATION_STEPS", "8"))  # Increase to compensate
NUM_EPOCHS = int(os.getenv("NUM_EPOCHS", "3"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "2e-4"))
MAX_SEQ_LENGTH = int(os.getenv("MAX_SEQ_LENGTH", "512"))
LORA_R = int(os.getenv("LORA_R", "16"))
LORA_ALPHA = int(os.getenv("LORA_ALPHA", "32"))
LORA_DROPOUT = float(os.getenv("LORA_DROPOUT", "0.05"))
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
HF_TOKEN = os.getenv("HF_TOKEN", os.getenv("HUGGING_FACE_TOKEN", None))

print(f"Model: {MODEL_NAME}")
print(f"Output: {OUTPUT_DIR}")
print(f"Using smaller model for local training (GPU memory constraints)")

In [ ]:
MODEL_NAME = os.getenv("MODEL_NAME", "meta-llama/Meta-Llama-3.1-8B")
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "./synova-finetuned")
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "4"))
GRADIENT_ACCUMULATION_STEPS = int(os.getenv("GRADIENT_ACCUMULATION_STEPS", "4"))
NUM_EPOCHS = int(os.getenv("NUM_EPOCHS", "3"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "2e-4"))
MAX_SEQ_LENGTH = int(os.getenv("MAX_SEQ_LENGTH", "512"))
LORA_R = int(os.getenv("LORA_R", "16"))
LORA_ALPHA = int(os.getenv("LORA_ALPHA", "32"))
LORA_DROPOUT = float(os.getenv("LORA_DROPOUT", "0.05"))
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
HF_TOKEN = os.getenv("HF_TOKEN", os.getenv("HUGGING_FACE_TOKEN", None))

print(f"Model: {MODEL_NAME}")
print(f"Output: {OUTPUT_DIR}")

## Load Dataset

In [ ]:
DATASET_FILE = Path("c:/Users/McBuz/CascadeProjects/Synova AI Rebuild/synova-workspace/data/training/synova_dataset.jsonl")

def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

if DATASET_FILE.exists():
    sample_data = load_jsonl(DATASET_FILE)
    print(f"Loaded {len(sample_data)} examples")
else:
    print("Using sample data")
    sample_data = [
        {"instruction": "What is PeakBrain?", "input": "", "output": "PeakBrain orchestrates Synova's AI components."},
        {"instruction": "How to add API endpoint?", "input": "", "output": "Create router in apps/api/src/routers/ and register in main.py."}
    ]

print(f"Total examples: {len(sample_data)}")

## Format Dataset

In [ ]:
def format_example(example):
    instruction = example["instruction"]
    input_text = example.get("input", "")
    output = example["output"]
    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}"
    return f"### Instruction:\n{instruction}\n\n### Response:\n{output}"

formatted_data = [format_example(ex) for ex in sample_data]
print(f"Formatted {len(formatted_data)} examples")

# Load Model with Quantization
try:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
        llm_int8_enable_fp32_cpu_offload=True
    )
    print("Model loaded with 4-bit quantization and CPU offload")
except Exception as e:
    print(f"4-bit failed: {e}, trying CPU-only with smaller model")
    # Use smaller model for CPU training
    MODEL_NAME = "meta-llama/Llama-3.2-3B"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cpu",
        trust_remote_code=True,
        token=HF_TOKEN
    )
    print(f"Loaded smaller model {MODEL_NAME} on CPU")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer loaded")

## Load Model with Quantization

In [ ]:
try:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN
    )
    print("Model loaded with 4-bit quantization")
except Exception as e:
    print(f"4-bit failed: {e}, trying 8-bit")
    quantization_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quantization_config, device_map="auto", trust_remote_code=True, token=HF_TOKEN)
    print("Model loaded with 8-bit quantization")

## Apply LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
print(f"Trainable parameters: {model.print_trainable_parameters()}")

## Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples, truncation=True, max_length=MAX_SEQ_LENGTH, padding="max_length", return_tensors=None)

tokenized_data = tokenize_function(formatted_data)
train_dataset = Dataset.from_dict(tokenized_data)

if len(train_dataset) > 10:
    train_dataset = train_dataset.train_test_split(test_size=0.1)
    print(f"Train: {len(train_dataset['train'])}, Test: {len(train_dataset['test'])}")
else:
    train_dataset = {"train": train_dataset, "test": train_dataset}
    print("Using full dataset for training")

## Initialize Trainer

In [ ]:
class TrainingArguments:
    def __init__(self, output_dir, num_train_epochs, per_device_train_batch_size, per_device_eval_batch_size, gradient_accumulation_steps, learning_rate, warmup_steps: int, logging_steps: int, save_steps: int, eval_steps: int, save_total_limit: int, fp16: bool, optim: str, report_to: str, load_best_model_at_end: bool):
        pass
BATCH_SIZE = None
NUM_EPOCHS = None
def OUTPUT_DIR():
    pass
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=100,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    load_best_model_at_end=True
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset["train"],
    eval_dataset=train_dataset["test"],
    data_collator=data_collator
)

print("Trainer initialized")

## Start Training

In [ ]:
print("Starting training...")
trainer.train()
print("Training complete!")

## Save Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")